# 🔍 AI-Based Fake News Detection System
### BTech Final Year Major Project
**Pipeline:** TF-IDF (Baseline) → BERT (Main Model) → API Verification

---
**Steps:**
1. Install dependencies
2. Mount Google Drive
3. Load & preprocess ISOT dataset
4. TF-IDF Baseline (Logistic Regression)
5. BERT Fine-tuning
6. Evaluation & Comparison
7. API Verification Layer (Google Fact Check + NewsAPI)
8. Final Prediction Pipeline
9. Save Model

## 📦 Cell 1 — Install Dependencies

In [ ]:
!pip install transformers==4.40.0 torch scikit-learn pandas numpy requests
!pip install datasets accelerate evaluate seaborn matplotlib
print('✅ All dependencies installed!')

## 💾 Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/FakeNewsDetector'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ Drive mounted. Model will be saved to: {SAVE_DIR}')

## 📂 Cell 3 — Download ISOT Dataset
> Dataset: **ISOT Fake News Dataset** (Fake.csv + True.csv)
> Source: Kaggle — `clmentbisaillon/fake-and-real-news-dataset`

In [ ]:
# Option A: Upload kaggle.json first, then run this
# from google.colab import files
# files.upload()  # upload your kaggle.json here
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d clmentbisaillon/fake-and-real-news-dataset
# !unzip -q fake-and-real-news-dataset.zip

# Option B: Manual upload
# from google.colab import files
# uploaded = files.upload()  # upload Fake.csv and True.csv manually

# Option C: Download from HuggingFace datasets (no login needed)
from datasets import load_dataset
import pandas as pd

print('Downloading GossipCop dataset from HuggingFace (no login required)...')
dataset = load_dataset('GonzaloA/fake_news')
train_df = pd.DataFrame(dataset['train'])
test_df  = pd.DataFrame(dataset['test'])
val_df   = pd.DataFrame(dataset['validation'])

df = pd.concat([train_df, test_df, val_df]).reset_index(drop=True)
df = df.rename(columns={'label': 'label'})
df['text'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
df = df[['text', 'label']].dropna()

print(f'✅ Dataset loaded: {len(df)} rows')
print(df['label'].value_counts())
df.head()

## 🔧 Cell 4 — Preprocessing & Train/Test Split

In [ ]:
import re
import numpy as np
from sklearn.model_selection import train_test_split

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)       # remove URLs
    text = re.sub(r'<.*?>', '', text)                  # remove HTML tags
    text = re.sub(r'[^a-z0-9\s]', '', text)           # remove special chars
    text = re.sub(r'\s+', ' ', text).strip()           # remove extra spaces
    return text

df['clean_text'] = df['text'].apply(clean_text)

# Use 30k samples for faster training (increase if needed)
df_sample = df.sample(n=min(30000, len(df)), random_state=42).reset_index(drop=True)

X = df_sample['clean_text'].tolist()
y = df_sample['label'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'✅ Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Label distribution — Train: {np.bincount(y_train)} | Test: {np.bincount(y_test)}')

## 📊 Cell 5 — TF-IDF Baseline Model

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print('Training TF-IDF + Logistic Regression baseline...')

tfidf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),
        stop_words='english',
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs'))
])

tfidf_pipeline.fit(X_train, y_train)
tfidf_preds = tfidf_pipeline.predict(X_test)
tfidf_acc = accuracy_score(y_test, tfidf_preds)

print(f'\n✅ TF-IDF Accuracy: {tfidf_acc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, tfidf_preds, target_names=['REAL', 'FAKE']))

# Confusion Matrix
cm = confusion_matrix(y_test, tfidf_preds)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['REAL','FAKE'], yticklabels=['REAL','FAKE'])
plt.title('TF-IDF Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 🤖 Cell 6 — BERT Dataset Class

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertForSequenceClassification

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Using device: {device}')
if device.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

MAX_LEN = 256   # 256 is faster than 512, still good for news headlines+body
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = NewsDataset(X_train, y_train, tokenizer, MAX_LEN)
test_dataset  = NewsDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'✅ Dataset ready — Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

## 🏋️ Cell 7 — BERT Fine-Tuning

In [ ]:
from transformers import get_linear_schedule_with_warmup

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2,
    output_attentions=False,
    output_hidden_states=False
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, eps=1e-8)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

train_losses = []
val_accuracies = []

for epoch in range(EPOCHS):
    # ---- TRAIN ----
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        if step % 100 == 0:
            print(f'  Epoch {epoch+1} | Step {step}/{len(train_loader)} | Loss: {loss.item():.4f}')

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)

    # ---- VALIDATION ----
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    val_acc = correct / total
    val_accuracies.append(val_acc)
    print(f'\n🔹 Epoch {epoch+1}/{EPOCHS} — Loss: {avg_loss:.4f} | Val Accuracy: {val_acc:.4f}\n')

print('✅ BERT Training Complete!')

## 📈 Cell 8 — Plot Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, EPOCHS+1), train_losses, marker='o', color='crimson')
ax1.set_title('BERT Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True)

ax2.plot(range(1, EPOCHS+1), val_accuracies, marker='o', color='steelblue')
ax2.set_title('BERT Validation Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.grid(True)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_curves.png', dpi=150)
plt.show()
print('✅ Curves saved to Drive')

## 📋 Cell 9 — BERT Full Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

bert_acc = accuracy_score(all_labels, all_preds)
print(f'✅ BERT Accuracy: {bert_acc:.4f}')
print('\nClassification Report:')
print(classification_report(all_labels, all_preds, target_names=['REAL', 'FAKE']))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['REAL','FAKE'], yticklabels=['REAL','FAKE'])
plt.title('BERT Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/bert_confusion_matrix.png', dpi=150)
plt.show()

## 📊 Cell 10 — TF-IDF vs BERT Comparison

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

metrics = {
    'Model':     ['TF-IDF + LR (Baseline)', 'BERT Fine-tuned'],
    'Accuracy':  [
        round(accuracy_score(y_test, tfidf_preds), 4),
        round(bert_acc, 4)
    ],
    'Precision': [
        round(precision_score(y_test, tfidf_preds), 4),
        round(precision_score(all_labels, all_preds), 4)
    ],
    'Recall':    [
        round(recall_score(y_test, tfidf_preds), 4),
        round(recall_score(all_labels, all_preds), 4)
    ],
    'F1 Score':  [
        round(f1_score(y_test, tfidf_preds), 4),
        round(f1_score(all_labels, all_preds), 4)
    ]
}

results_df = pd.DataFrame(metrics)
print('\n📊 Model Comparison:')
print(results_df.to_string(index=False))

# Bar chart
x = np.arange(2)
width = 0.2
fig, ax = plt.subplots(figsize=(10, 5))
for i, metric in enumerate(['Accuracy', 'Precision', 'Recall', 'F1 Score']):
    ax.bar(x + i*width, results_df[metric], width, label=metric)
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(['TF-IDF + LR', 'BERT'])
ax.set_ylim(0.8, 1.0)
ax.set_title('Model Performance Comparison')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/model_comparison.png', dpi=150)
plt.show()

## 🌐 Cell 11 — API Verification Layer
> Uses **Google Fact Check Tools API** + **NewsAPI** for real-world claim verification
> Get free keys at: https://developers.google.com/fact-check/tools/api and https://newsapi.org

In [ ]:
import requests

# ⚠️ Add your API keys here
GOOGLE_FACTCHECK_API_KEY = 'YOUR_GOOGLE_API_KEY'
NEWS_API_KEY             = 'YOUR_NEWSAPI_KEY'

def verify_with_google_factcheck(claim: str):
    """Query Google Fact Check Tools API for existing fact checks."""
    try:
        url = 'https://factchecktools.googleapis.com/v1alpha1/claims:search'
        params = {'query': claim[:200], 'key': GOOGLE_FACTCHECK_API_KEY}
        response = requests.get(url, params=params, timeout=10)
        data = response.json()

        results = []
        for item in data.get('claims', [])[:3]:
            for review in item.get('claimReview', []):
                results.append({
                    'claim':     item.get('text', ''),
                    'rating':    review.get('textualRating', 'N/A'),
                    'publisher': review.get('publisher', {}).get('name', 'N/A'),
                    'url':       review.get('url', '')
                })
        return results if results else [{'message': 'No fact checks found'}]
    except Exception as e:
        return [{'error': str(e)}]

def verify_with_newsapi(keyword: str):
    """Search recent news articles related to the claim."""
    try:
        url = 'https://newsapi.org/v2/everything'
        # Extract first 5 words as keyword for search
        search_term = ' '.join(keyword.split()[:6])
        params = {
            'q':        search_term,
            'sortBy':   'relevancy',
            'pageSize': 3,
            'language': 'en',
            'apiKey':   NEWS_API_KEY
        }
        response = requests.get(url, params=params, timeout=10)
        articles = response.json().get('articles', [])
        return [{
            'title':  a.get('title', ''),
            'source': a.get('source', {}).get('name', ''),
            'url':    a.get('url', ''),
            'date':   a.get('publishedAt', '')[:10]
        } for a in articles]
    except Exception as e:
        return [{'error': str(e)}]

print('✅ API verification functions ready')
print('⚠️  Add your API keys above before running Cell 12')

## 🔮 Cell 12 — Final Prediction Pipeline

In [ ]:
def predict_news(text: str, use_api: bool = True):
    """
    Full pipeline: TF-IDF + BERT + API Verification
    Returns: verdict dict with confidence scores and sources
    """
    clean = clean_text(text)

    # --- TF-IDF prediction ---
    tfidf_prob  = tfidf_pipeline.predict_proba([clean])[0][1]

    # --- BERT prediction ---
    model.eval()
    enc = tokenizer(
        clean,
        truncation=True,
        padding='max_length',
        max_length=MAX_LEN,
        return_tensors='pt'
    )
    with torch.no_grad():
        outputs = model(
            input_ids=enc['input_ids'].to(device),
            attention_mask=enc['attention_mask'].to(device)
        )
    bert_probs = torch.softmax(outputs.logits, dim=1)[0]
    bert_prob  = bert_probs[1].item()   # prob of FAKE

    # --- Weighted ensemble (BERT dominant) ---
    final_score = 0.25 * tfidf_prob + 0.75 * bert_prob

    # --- Verdict ---
    if final_score >= 0.65:
        verdict = 'FAKE'
        confidence = final_score
    elif final_score <= 0.35:
        verdict = 'REAL'
        confidence = 1 - final_score
    else:
        verdict = 'UNCERTAIN'
        confidence = 1 - abs(0.5 - final_score) * 2

    result = {
        'verdict':           verdict,
        'confidence':        round(confidence * 100, 2),
        'tfidf_fake_prob':   round(tfidf_prob * 100, 2),
        'bert_fake_prob':    round(bert_prob  * 100, 2),
        'ensemble_score':    round(final_score * 100, 2),
        'fact_checks':       [],
        'related_articles':  []
    }

    # --- API Verification ---
    if use_api:
        print('  Querying fact-check APIs...')
        result['fact_checks']      = verify_with_google_factcheck(text)
        result['related_articles'] = verify_with_newsapi(text)

    return result

# -------- TEST IT --------
sample_news = "Scientists discover that drinking coffee every day doubles your lifespan, Harvard study confirms."

print(f'\nTesting on: "{sample_news}"\n')
result = predict_news(sample_news, use_api=False)  # set True if API keys added

print('=' * 50)
print(f'  VERDICT     : {result["verdict"]}')
print(f'  CONFIDENCE  : {result["confidence"]}%')
print(f'  TF-IDF Fake : {result["tfidf_fake_prob"]}%')
print(f'  BERT Fake   : {result["bert_fake_prob"]}%')
print(f'  Ensemble    : {result["ensemble_score"]}%')
print('=' * 50)

## 💾 Cell 13 — Save Model to Google Drive

In [ ]:
import pickle

# Save BERT model
model.save_pretrained(f'{SAVE_DIR}/bert_model')
tokenizer.save_pretrained(f'{SAVE_DIR}/bert_tokenizer')

# Save TF-IDF pipeline
with open(f'{SAVE_DIR}/tfidf_pipeline.pkl', 'wb') as f:
    pickle.dump(tfidf_pipeline, f)

# Save results
results_df.to_csv(f'{SAVE_DIR}/model_comparison.csv', index=False)

print('✅ All models saved to Google Drive!')
print(f'   📁 {SAVE_DIR}/')
print('   ├── bert_model/')
print('   ├── bert_tokenizer/')
print('   ├── tfidf_pipeline.pkl')
print('   ├── model_comparison.csv')
print('   ├── training_curves.png')
print('   ├── bert_confusion_matrix.png')
print('   └── model_comparison.png')

## 🧪 Cell 14 — Batch Test on Custom Examples

In [ ]:
test_headlines = [
    "NASA confirms water found on Mars in large quantities.",
    "Government secretly adding chemicals to drinking water to control population.",
    "Stock market hits record high as economy shows strong growth.",
    "Eating 5G chips via vaccines causes cancer, whistleblower reveals.",
    "WHO releases updated guidelines on COVID-19 vaccination."
]

print('\n🔍 Batch Prediction Results:\n')
print(f'{"Headline":<65} {"Verdict":<12} {"Confidence"}')
print('-' * 90)

for headline in test_headlines:
    res = predict_news(headline, use_api=False)
    short = headline[:62] + '...' if len(headline) > 62 else headline
    emoji = '🔴' if res['verdict'] == 'FAKE' else ('🟢' if res['verdict'] == 'REAL' else '🟡')
    print(f'{short:<65} {emoji} {res["verdict"]:<10} {res["confidence"]}%')